# Medical Cost Regression

**Tabular Regression Project** — Predict annual medical cost from patient demographics and health risk factors.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `medical_cost`

## 2 · Project Overview

We predict **annual medical costs** from patient demographics, lifestyle factors, and health risk indicators. This classic insurance/healthcare regression task helps actuaries set premiums and health systems identify high-cost patients for intervention programs.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given a patient's age, gender, BMI, smoking status, number of children, region, exercise frequency, chronic conditions, income, and insurance plan, predict the **annual medical cost in USD**.

## 5 · Why This Project Matters

- **Insurance pricing** relies on accurate cost prediction for premium setting.
- **Population health management** targets high-cost patients for preventive interventions.
- Understanding the **smoker × BMI interaction** reveals multiplicative risk.
- **Exercise and lifestyle** factors show the financial value of prevention.
- A foundational dataset for learning healthcare analytics.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | 10 (age, gender, BMI, smoker, children, region, exercise, chronic conditions, income, insurance plan) |
| **Target** | `medical_cost` (continuous, USD/year) |
| **Categorical** | region (4), insurance_plan (4) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "medical_cost"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8000
age = np.random.randint(18, 75, n)
gender = np.random.choice([0, 1], n)
bmi = np.random.normal(28, 6, n).clip(15, 55)
smoker = np.random.choice([0, 1], n, p=[0.75, 0.25])
children = np.random.randint(0, 6, n)
region = np.random.choice(["northeast", "northwest", "southeast", "southwest"], n)
exercise_freq = np.random.randint(0, 7, n)
chronic_conditions = np.random.choice([0, 1, 2, 3], n, p=[0.4, 0.3, 0.2, 0.1])
annual_income_k = np.random.uniform(20, 150, n)
insurance_plan = np.random.choice(["bronze", "silver", "gold", "platinum"], n, p=[0.3, 0.35, 0.25, 0.1])

cost = (age * 200 + bmi * 100
        + smoker * 20000 + children * 500
        + chronic_conditions * 5000
        - exercise_freq * 300
        + annual_income_k * 20
        + np.random.normal(0, 2000, n))

# Smoker × BMI interaction (obese smokers have multiplicative risk)
cost += smoker * np.maximum(bmi - 30, 0) * 500
cost = np.maximum(cost, 500)

df = pd.DataFrame({
    "age": age, "gender": gender, "bmi": bmi,
    "smoker": smoker, "children": children,
    "region": region, "exercise_freq_weekly": exercise_freq,
    "chronic_conditions": chronic_conditions,
    "annual_income_k": annual_income_k,
    "insurance_plan": insurance_plan,
    "medical_cost": cost
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["age", "bmi", "children",
                          "exercise_freq_weekly", "chronic_conditions", "annual_income_k"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["medical_cost"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("medical_cost"); ax.set_title(f"Cost vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("smoker")["medical_cost"].mean().plot(
    kind="bar", ax=axes[0], color=["steelblue", "coral"], edgecolor="black")
axes[0].set_title("Mean Cost: Smoker vs Non-Smoker"); axes[0].set_ylabel("$")
axes[0].set_xticklabels(["Non-Smoker", "Smoker"], rotation=0)
df.groupby("insurance_plan")["medical_cost"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Mean Cost by Insurance Plan"); axes[1].set_xlabel("$")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `medical_cost`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("$")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Std: ${df[TARGET].std():,.0f}, Range: ${df[TARGET].min():,.0f} - ${df[TARGET].max():,.0f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["smoker_bmi"] = X_train["smoker"] * X_train["bmi"]
X_test["smoker_bmi"] = X_test["smoker"] * X_test["bmi"]

X_train["age_chronic"] = X_train["age"] * X_train["chronic_conditions"]
X_test["age_chronic"] = X_test["age"] * X_test["chronic_conditions"]

X_train["health_risk_score"] = X_train["smoker"] * 10 + X_train["chronic_conditions"] * 5 + np.maximum(X_train["bmi"] - 25, 0)
X_test["health_risk_score"] = X_test["smoker"] * 10 + X_test["chronic_conditions"] * 5 + np.maximum(X_test["bmi"] - 25, 0)

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Smoking status** is the single strongest predictor — smokers cost ~$20K+ more per year.
- **Smoker × BMI interaction** reveals that obese smokers face multiplicative cost increases.
- **Chronic conditions** add ~$5K per condition to annual costs.
- **Age** has a steady positive effect (~$200/year per year of age).
- **Exercise frequency** provides a modest but consistent cost reduction.

**Business takeaway:** Smoking cessation programs are the highest-ROI health intervention. For obese smokers, the combined risk is far greater than the sum of individual risks — targeted programs for this group have outsized impact.

## 26 · Limitations

1. Synthetic data — real medical costs follow complex distributions with extreme outliers.
2. No diagnosis-level detail — chronic_conditions is a coarse proxy.
3. No temporal component — costs change year-to-year.
4. No prescription drug costs or specific procedure costs.
5. Insurance plan type may be endogenous (sicker people choose better plans).

## 27 · How to Improve This Project

1. Use real insurance claims data (CMS, commercial payers).
2. Add diagnosis codes (ICD-10) and procedure codes (CPT).
3. Include prescription drug utilization.
4. Model cost as a two-part model: P(any cost) × E(cost|cost > 0).
5. Add longitudinal data for cost trajectory prediction.

## 28 · Production Considerations

- Deploy for real-time premium estimation at enrollment.
- Integrate with EHR for risk stratification.
- Monitor for adverse selection patterns.
- Update annually with claims experience data.

## 29 · Common Mistakes

1. Missing the smoker × BMI interaction — linear models miss this multiplicative effect.
2. Treating the right-skewed cost distribution as normal.
3. Not separating high-cost outliers (catastrophic claims) from routine costs.
4. Using insurance plan type without controlling for selection bias.
5. Ignoring that exercise frequency is self-reported and may be inaccurate.

## 30 · Mini Challenge / Exercises

1. Remove `smoker` — how much does R² drop?
2. Compare the model with and without the `smoker_bmi` interaction term.
3. Try log(medical_cost) as the target — does it improve residual normality?
4. Build separate models for smokers and non-smokers — compare RMSE.
5. What BMI threshold maximizes the smoker × BMI interaction effect?

## 31 · Final Summary / Key Takeaways

1. **Smoking status** is by far the strongest cost predictor.
2. **Smoker × BMI interaction** reveals multiplicative, not additive, risk.
3. **Chronic conditions** and **age** drive baseline cost escalation.
4. **Gradient boosting** captures the non-linear interactions that linear models miss.
5. **Feature engineering** (smoker_bmi, health risk score) encodes domain knowledge.
6. **Healthcare costs** are right-skewed — log-transform or two-part models may help.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))